# kodoom: CPU preflight, then resumable Gemma 3 27B translation

First select **Accelerator=None**, enable Internet, attach the version-1 Flax model,
and leave **RUN_TPU=False**. Run all or the first two code cells on CPU until
**CPU PREFLIGHT PASSED**. Missing secrets/access now fail before TPU setup.

Then stop the CPU session, select **TPU v5e-8**, set **RUN_TPU=True**, and use
**Save Version → Save & Run All**. The saved session repeats the CPU checks.
CPU checks do not certify TPU imports, 27B memory fit, speed or translation quality.

Active data recipe: **bulk-translation**, project-plan **v21**, section 1.2;
CPU-first handoff: project-plan **v24**. Generate unreviewed drafts of all
**1,200 train + 400 test typed cases (8,000 decisions)** plus **2,000 helmo records**
with **gemma3-27b-tpu-bf16**, ending at **drafts-complete-awaiting-review**.

Use a **private** notebook. Accept Gemma terms on Kaggle and attach
`google/gemma-3/flax/gemma3-27b-it/1` under Inputs/Models. Create an existing
**private HF dataset repo** and enable its write token as the **HF_TOKEN** secret
for this notebook. **HF_TOKEN** is for dataset read/write access; Gemma access
is granted on Kaggle for the attached model. This notebook fetches public GitHub
code anonymously and ignores **GITHUB_TOKEN**. Never paste credentials into cells.

Keep campaign/storage settings stable on resume. A changed code pin is rejected
for an old saved campaign unless deliberately migrated; this CPU check detects it.
See `docs/kaggle-huggingface-setup-guide.md` and `docs/execution-workflow.md`.


In [ ]:
PROVIDER = "kaggle"
PROFILE = "kaggle-tpu"
BACKEND = "jax"
CODE_REVISION = "fbb73579ed31c8959a056b2aa95233e23b2d0edb"
CHECKOUT = "/tmp/kodoom-tpu-code"
CAMPAIGN_ID = "gemma27b-bulk-v1"
HF_DATASET_REPO = ""  # REQUIRED: your private organization/dataset repo
OPERATOR = "owner"  # change this name for an authorized collaborator
RUN_TPU = False  # first run on CPU; True only after CPU PREFLIGHT PASSED
TAKEOVER = False  # True only after confirming an interrupted prior worker ended

## CPU checks before entering the TPU queue

Run this cell with **Accelerator=None** and Internet enabled. It checks Python,
GitHub checkout, HF secret availability, private dataset read/write/readback,
mounted Flax model access, source inputs, prompt lengths, writable paths,
resume conflicts and pinned dependency resolution. Lightweight tools use an
isolated CPU environment; weights and accelerator packages are not installed.
Dependency resolution may download package metadata/wheels and take a few minutes.

A unique tiny private HF probe file is added, read back and removed; its two
commits remain in repository history. No campaign writer lease is acquired.
Model access is checked through the mounted Kaggle asset, not an HF model license.
A missing secret must be created AND enabled for this notebook under Add-ons > Secrets.

Environment setup does not require provider ensurepip; it repairs a partial venv
and installs pip using the notebook pip before adding lightweight CPU tools.


In [ ]:
import sys
import time
import types
import urllib.parse
import urllib.request

SESSION_STARTED = time.monotonic()
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Use a Kaggle Linux runtime with Python 3.11 through 3.13")
if not HF_DATASET_REPO:
    raise ValueError("Set HF_DATASET_REPO before running CPU checks")


# Public source: do not send an optional stale GitHub credential.
url = (
    "https://raw.githubusercontent.com/Farahani1/kodum/"
    + urllib.parse.quote(CODE_REVISION, safe="")
    + "/src/kodoom/runtime.py"
)
with urllib.request.urlopen(url, timeout=30) as response:
    source = response.read().decode("utf-8")
runtime = types.ModuleType("kodoom_bootstrap")
exec(compile(source, "runtime.py", "exec"), runtime.__dict__)
# The pinned runtime also reads GitHub secrets during CPU and TPU checkout.
# Disable that credential here while preserving its HF secret adapter.
runtime.secret = lambda provider, name, read=runtime.secret: (
    None if name == "GITHUB_TOKEN" else read(provider, name)
)
preflight = runtime.cpu_preflight(
    PROVIDER,
    CODE_REVISION,
    CHECKOUT,
    campaign_id=CAMPAIGN_ID,
    repo=HF_DATASET_REPO,
    operator=OPERATOR,
    takeover=TAKEOVER,
    profile=PROFILE,
    backend=BACKEND,
)

## Explicit TPU opt-in: restore, benchmark and translate

With **RUN_TPU=False**, this cell skips accelerator installation and translation.
After CPU preflight passes, stop the CPU session, select TPU v5e-8 and set
**RUN_TPU=True** before submitting Save Version > Save & Run All.

The TPU run repeats the checks, installs its pinned accelerator dependencies,
probes eight devices and loads BF16 weights once. Its eight-hour budget starts
in the early setup cell; 30 minutes are reserved for finalization.
BF16 batches 1/2/4/8 are benchmarked and frozen, snapshots saved about every ten
minutes, and only complete cases are saved. Resumes reuse the production identity.


In [ ]:
result = None
if RUN_TPU:
    if not preflight["passed"]:
        raise RuntimeError("CPU preflight must pass before TPU setup")
    checkout = runtime.bootstrap(PROVIDER, CODE_REVISION, CHECKOUT, backend=BACKEND)
    from kodoom.bulk.launch import run

    result = run(
        provider=PROVIDER,
        profile=PROFILE,
        campaign_id=CAMPAIGN_ID,
        repo=HF_DATASET_REPO,
        operator=OPERATOR,
        takeover=TAKEOVER,
        session_started=SESSION_STARTED,
    )
else:
    print("CPU-only mode: TPU installation, model loading and translation skipped.")

## Progress and review

Remote `campaigns/<campaign-id>/checkpoint.json` identifies a verified archive
with English inputs, complete-case shards, progress, events, production settings,
TPU metrics and review CSVs. Local artifacts are under
`/kaggle/working/kodoom/data/bulk/<campaign-id>/`. Save private notebook outputs
as an additional copy. Keep model weights and caches out of saved output.

Review `review/cases-*.csv` and `review/labels-*.csv` with the English/Persian
context. Empty review fields remain unreviewed; automatic findings are flags.
All 400 test cases need human review. A diagnostic sample cannot establish
dataset-wide label error prevalence or detector coverage. Adoption, training
and publication still require the research decisions in the project plan.

Read-only snapshot download and pause instructions are in `docs/execution-workflow.md`.
Actual TPU fit, speed and a second live session remain unverified until run.


In [ ]:
if result is None:
    print("CPU preflight report:", "/kaggle/working/kodoom/cpu-preflight.json")
    print("After PASS: stop CPU session, select TPU v5e-8, set RUN_TPU=True, Save & Run All.")
else:
    print("Status:", result["status"])
    print("Completed / failed / total cases:", result["counts"])
    print("Private artifacts:", result["artifact_root"])
    print("Last verified remote revision:", result["last_verified_revision"])
    print("Final remote save verified:", result["persisted"])
    print("Stop here for human review; resume the same campaign if work remains.")